# doc-agent — End-to-End Pipeline Demo (Stage 1-4)

Mirrors the shape of your teammate's `kb_demo.ipynb`-style single-page test, but wired to
**your** codebase (`pipeline.py`'s fixed Stage 0-9 order):

```
pages -> preprocess -> enhance -> layout -> OCR -> chunk -> embed -> store
```

Differences from your teammate's notebook (so nothing here silently does the wrong thing):
- Your OCR backend is **RapidOCR (PP-OCRv4 ONNX)**, not surya — set in `configs/config.yaml` ->
  `ocr.model`. No `SURYA_INFERENCE_BACKEND` / `TORCH_DEVICE` env-var juggling is needed.
- Your pipeline has an extra **preprocess** (deskew/denoise/binarize) and **enhance**
  (CLAHE / self-supervised UNet) stage *before* layout detection — `pipeline.py` runs both,
  so this notebook does too, or `_page_image()` inside `ocr.py` won't find the image it expects.
- This notebook runs on **one held-out sample page** from `grading_kit/heldout_pages/`
  (already in the repo) so it needs **no corpus download**. The full-corpus run is one command,
  shown at the bottom, that you run yourself when you're ready.

**Before running:** make sure deps are installed — from the repo root:
```bash
uv sync --frozen          # or: pip install -e . --break-system-packages
```


In [ ]:



import os
import sys
from pathlib import Path

# Keep thread usage sane in Jupyter (mirrors the intent of your teammate's notebook,
# but there's no SURYA_/TORCH_DEVICE overrides to clear here — this codebase doesn't use them)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")

# This notebook lives in notebooks/, so the repo root is one level up
ROOT_DIR = Path().resolve().parent
SRC_DIR = ROOT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Every relative path in configs/config.yaml (data/raw, data/index, data/interim/...)
# is resolved relative to the CWD, so run everything from the repo root.
os.chdir(ROOT_DIR)
print("cwd:", Path().resolve())

from doc_agent import config
from doc_agent.contracts import Page
from doc_agent.ingest import preprocess, enhance
from doc_agent.vision import layout, ocr
from doc_agent.index import chunk, embed, store


cwd: E:\CSE 429\doc-agent-G18


In [2]:
cfg = config.load()  # reads configs/config.yaml — the single source of truth for every stage

# configs/config.yaml defaults to device: cuda; every stage (ocr, embed, enhance) auto-falls back
# to cpu when no GPU / no CUDA execution provider is available, so you normally don't need to touch
# this. Uncomment to force CPU explicitly (e.g. for a quick laptop demo):
# cfg["device"] = "cpu"

print("device:       ", cfg["device"])
print("preprocess:   ", cfg["preprocess"])
print("enhance model:", cfg["enhance"]["model"])
print("layout model: ", cfg["layout"]["model"])
print("ocr model:    ", cfg["ocr"]["model"])
print("embed model:  ", cfg["embed"]["model"], "dim:", cfg["embed"]["dim"])
print("index type:   ", cfg["index"]["type"])


device:        cuda
preprocess:    {'enabled': True, 'denoise': True, 'binarize': True, 'deskew': True, 'augment': False}
enhance model: clahe_denoise
layout model:  heuristic:projection
ocr model:     rapidocr:PP-OCRv4-onnx
embed model:   sentence-transformers/all-MiniLM-L6-v2 dim: 384
index type:    faiss:hnsw


## 1. Pick a test page

No need to run `scripts/get_data.sh` (which pulls the ~126MB corpus PDF) just to sanity-check
the pipeline. `grading_kit/heldout_pages/` already ships a few real scanned pages from the same
corpus — use one of those.


In [3]:
HELDOUT_DIR = ROOT_DIR / "grading_kit" / "heldout_pages"
test_image_path = HELDOUT_DIR / "bengal1889_p0018.jpg"
assert test_image_path.exists(), f"missing sample page: {test_image_path}"

# id must look like <doc_id>_p<NNNN> — ocr.py derives doc_id from it via page_id.split('_p')[0]
test_page = Page(id="bengal1889_p0018", doc_id="bengal1889", image_path=str(test_image_path))

print(f"Testing pipeline on {test_image_path.name} ...", flush=True)


Testing pipeline on bengal1889_p0018.jpg ...


## 2. Preprocess + Enhance (Stage 1)

`pipeline.build_knowledge_base()` runs these *before* layout detection, and `ocr.transcribe()`
later looks for the page image under `data/interim/enhanced/` first — so skipping this stage
would make OCR fail to find the image. Run it here too, on our one-page list.


In [4]:
print("Running preprocess (deskew / denoise / binarize)...", flush=True)
pages = preprocess.run([test_page], cfg)
print(f"  -> {pages[0].image_path}")

print("Running enhance (CLAHE + unsharp, or the trained unet_small)...", flush=True)
pages = enhance.run(pages, cfg)
print(f"  -> {pages[0].image_path}")


Running preprocess (deskew / denoise / binarize)...
{"ts":"2026-08-15 23:15:01,434","lvl":"INFO","mod":"ingest.preprocess","msg":"preprocessed 1 pages"}
  -> data\interim\processed\bengal1889_p0018.jpg
Running enhance (CLAHE + unsharp, or the trained unet_small)...
{"ts":"2026-08-15 23:15:01,434","lvl":"WARNING","mod":"ingest.enhance","msg":"device: cuda requested but torch has no CUDA — enhancer falls back to cpu"}
  -> data\interim\enhanced\bengal1889_p0018.jpg


## 3. Layout detection + OCR (Stages 2-3)

In [5]:
print("Running layout detection...", flush=True)
regions = layout.detect(pages, cfg)
print(f"Found {len(regions)} structural regions.\n")
for r in regions[:8]:
    print(f"  {r.kind:8s} bbox={r.bbox}")

print("\nRunning OCR & region-level chunking...", flush=True)
ocr_chunks = ocr.transcribe(regions, cfg)
print(f"Generated {len(ocr_chunks)} text chunks.\n")

print("--- EXTRACTED CHUNKS (first 3) ---")
for c in ocr_chunks[:3]:
    print(f"[{c.id}]")
    print(c.text[:400] + ("..." if len(c.text) > 400 else ""))
    print("-" * 40)


Running layout detection...
{"ts":"2026-08-15 23:15:01,609","lvl":"INFO","mod":"vision.layout","msg":"layout: 3 regions from 1 pages"}
Found 3 structural regions.

  text     bbox=(262, 170, 1370, 1822)
  heading  bbox=(827, 106, 842, 135)
  text     bbox=(105, 1858, 1512, 2306)

Running OCR & region-level chunking...
{"ts":"2026-08-15 23:15:01,619","lvl":"INFO","mod":"vision.ocr","msg":"ocr: 3 chunks from 3 regions"}
Generated 3 text chunks.

--- EXTRACTED CHUNKS (first 3) ---
[bengal1889_p0018:r0]
issued by Government in1888,drawing the attention of all local offoers,and
especially of Municipal Commissioners, to the necessity of improving the regis
tration of vital statistics,has had the desired effect of stimulating the energies
of the authorities for whom it wawinteuded;and although in some districts
and towns near the bottom of the list it is evident that registration is still
neglected,...
----------------------------------------
[bengal1889_p0018:r1]

---------------------------

## 4. Chunk + Embed (Stage 4)

`chunk.split()` re-windows each OCR'd region to the embedding model's own tokenizer
(`cfg['index']['chunk_tokens']` / `overlap`) without crossing region boundaries.
`embed.encode()` then embeds every window with `sentence-transformers/all-MiniLM-L6-v2`.


In [6]:
print("Re-chunking to embedding-model token windows...", flush=True)
final_chunks = chunk.split(ocr_chunks, cfg)
print(f"{len(ocr_chunks)} OCR region-chunks -> {len(final_chunks)} token-windowed chunks")

print("\nEmbedding chunks...", flush=True)
vectors = embed.encode(final_chunks, cfg)
print(f"Embedded {len(vectors)} chunks, dim={len(vectors[0]) if vectors else 0}")


Re-chunking to embedding-model token windows...


c:\Users\Rubaiyat\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (877 > 512). Running this sequence through the model will result in indexing errors


3 OCR region-chunks -> 8 token-windowed chunks

Embedding chunks...


c:\Users\Rubaiyat\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Embedded 8 chunks, dim=384


## 5. Store (optional, this-page-only demo index)

Writes to `data/index_demo/` — a separate directory from `data/index/` — so this demo never
clobbers your real corpus index (the one `notebooks/kb_demo.ipynb` reads and the one the
grading kit checks).


In [10]:
demo_cfg = {**cfg, "index": {**cfg["index"], "dir": "data/index_demo"}}

store.build(final_chunks, vectors, demo_cfg)
print("Wrote demo index to", demo_cfg["index"]["dir"])

demo_index, loaded_chunks = store.load(demo_cfg)
print(f"Reloaded index: {demo_index.ntotal} vectors, {len(loaded_chunks)} chunk records")

for i in range(len(loaded_chunks)):
    print(f"[{loaded_chunks[i].id}] {loaded_chunks[i].text}...")



Wrote demo index to data/index_demo
Reloaded index: 8 vectors, 8 chunk records
[bengal1889_p0018:r0-0] issued by government in1888, drawing the attention of all local offoers, and especially of municipal commissioners, to the necessity of improving the regis tration of vital statistics, has had the desired effect of stimulating the energies of the authorities for whom it wawinteuded ; and although in some districts and towns near the bottom of the list it is evident that registration is still neglected, there is every reason to be satisfied with the progress made during the year under review in the province taken as a whole. in 28 out of the 45districts there was an improvement in registration in 1889, as compared with 1888 and the average of the five years 1884 - 88. theimprovement was most marked in the districts of poori, balasore, noakhali, purheah, cuttack, and serampore. according to the reports of the local officers the healthof thosedistrictsduring1889wasdecidedlybad, andworse 

## Running this on the full corpus

This notebook only touched one held-out page, so nothing was downloaded. To build the real
knowledge base over the whole scanned corpus, run (outside this notebook, since it fetches a
~126MB PDF and OCRs hundreds of pages):

```bash
bash scripts/get_data.sh      # fetches the corpus PDF into data/raw/ (gitignored)
make ingest index             # == python scripts/run_ingest.py -> pipeline.build_knowledge_base(cfg)
```

or equivalently from Python:

```python
from doc_agent import config, pipeline
pipeline.build_knowledge_base(config.load())
```

That runs the *fixed* Stage 0-9 order in `pipeline.py` (load -> preprocess -> enhance -> layout
-> OCR -> chunk -> embed -> store) over every page under `data/raw/`, and writes the real index
to `data/index/` — the same files `notebooks/kb_demo.ipynb` reports statistics on.
